In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
google_api_key = os.getenv("GOOGLE_API_KEY")


In [3]:
if google_api_key == "":
    print("Google API key is not set. Please set the GOOGLE_API_KEY environment variable.")
else:
    print("Google API key is set.")

Google API key is set.


In [5]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext, StorageContext, load_index_from_storage
from llama_index.llms.gemini import Gemini
import google.generativeai as genai
from llama_index.embeddings.gemini import GeminiEmbedding
from IPython.display import Markdown, display


In [6]:
genai.configure(api_key=google_api_key)

In [8]:
for models in genai.list_models():
    print(models)

Model(name='models/gemini-2.5-flash',
      base_model_id='',
      version='001',
      display_name='Gemini 2.5 Flash',
      description=('Stable version of Gemini 2.5 Flash, our mid-size multimodal model that '
                   'supports up to 1 million tokens, released in June of 2025.'),
      input_token_limit=1048576,
      output_token_limit=65536,
      supported_generation_methods=['generateContent',
                                    'countTokens',
                                    'createCachedContent',
                                    'batchGenerateContent'],
      temperature=1.0,
      max_temperature=2.0,
      top_p=0.95,
      top_k=64)
Model(name='models/gemini-2.5-pro',
      base_model_id='',
      version='2.5',
      display_name='Gemini 2.5 Pro',
      description='Stable release (June 17th, 2025) of Gemini 2.5 Pro',
      input_token_limit=1048576,
      output_token_limit=65536,
      supported_generation_methods=['generateContent',
                  

In [11]:
document = SimpleDirectoryReader("../Data").load_data()

In [15]:
print(document[0].text)

The History of Space Exploration

Space exploration began in earnest in the mid-20th century during the Cold War between the United States and the Soviet Union. Both nations raced to achieve milestones in space, leading to some of humanity's greatest achievements.

The Soviet Union launched Sputnik 1 on October 4, 1957, making it the first artificial satellite to orbit Earth. This shocked the world and marked the beginning of the space age. Shortly after, the Soviets sent the first living creature to space, a dog named Laika, aboard Sputnik 2 in November 1957.

Yuri Gagarin became the first human to travel to space on April 12, 1961, completing one orbit around Earth aboard Vostok 1. His flight lasted 108 minutes and made him a global hero.

The United States responded with the Apollo program. On July 20, 1969, NASA astronauts Neil Armstrong and Buzz Aldrin became the first humans to walk on the Moon during the Apollo 11 mission. Armstrong's words, "That's one small step for man, one g

In [23]:
model = Gemini(model_name="models/gemma-4-26b-a4b-it")

C:\Users\USER\AppData\Local\Temp\ipykernel_23612\3709555656.py:1: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  model = Gemini(model_name="models/gemma-4-26b-a4b-it")


In [20]:
gemini_embed_model = GeminiEmbedding(model_name="models/gemini-embedding-001")

C:\Users\USER\AppData\Local\Temp\ipykernel_23612\2522577506.py:1: DeprecationWarning: Call to deprecated class GeminiEmbedding. (Should use `llama-index-embeddings-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/embeddings/google_genai/Support for this package will be discontinued as of v0.4.2) -- Deprecated since version 0.4.2.
  gemini_embed_model = GeminiEmbedding(model_name="models/gemini-embedding-001")


In [18]:
for model in genai.list_models():
    if 'embedContent' in model.supported_generation_methods:
        print(model.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [22]:
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-resea

In [25]:
from llama_index.core import Settings

Settings.llm = model
Settings.embed_model = gemini_embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

In [26]:
index = VectorStoreIndex.from_documents(document)

In [27]:
index.storage_context.persist(persist_dir="./storage")

In [30]:
query_engine = index.as_query_engine()

In [32]:
print(query_engine.query("who is rahul yadav?"))

*   Role: Expert Q&A system.
    *   Constraint 1: Use *only* the provided context.
    *   Constraint 2: Do *not* use prior knowledge.
    *   Constraint 3: Never directly reference the context (e.g., "Based on the context...").
    *   Query: "who is rahul yadav?"

    *   The text discusses the history of space exploration, Sputnik 1, Laika, Yuri Gagarin, Apollo program, Neil Armstrong, Buzz Aldrin, Mir, ISS, SpaceX, Elon Musk, NASA's Perseverance rover, and the James Webb Space Telescope.

    *   The name "Rahul Yadav" does not appear anywhere in the provided text.

    *   Since the name "Rahul Yadav" is not mentioned in the context, and I am forbidden from using prior knowledge, I must state that the information is not available in the provided text.

    *   "The provided information does not contain any information about Rahul Yadav." (Wait, rule 1 says "Never directly reference the given context in your answer." and rule 2 says "Avoid statements like 'Based on the context...'

In [29]:
summary = index.as_chat_engine().chat("give me a summary of the document")
print(summary)


*   Context: Two snippets from a file named "The History of Space Exploration.txt".
    *   Task: Provide a detailed summary of the document.
    *   Constraint: Use only the provided documents. If information is missing, say "don't know". Be talkative and provide specific details.

    *   *Snippet 1:* Mentions James Webb Space Telescope (launched Dec 2021, images of galaxies from 13 billion years ago). Mentions daily technologies from space (GPS, weather forecasting, satellite internet, memory foam, water filters, scratch-resistant lenses). Mentions future plans (Moon bases, Mars missions, asteroid mining).
    *   *Snippet 2 (Main Text):*
        *   Origins: Mid-20th century, Cold War (US vs. USSR).
        *   Key Milestones (USSR): Sputnik 1 (Oct 4, 1957 - first artificial satellite); Sputnik 2 (Nov 1957 - Laika the dog); Yuri Gagarin (April 12, 1961 - first human, Vostok 1, 108 minutes).
        *   Key Milestones (USA): Apollo program; Apollo 11 (July 20, 1969 - Neil Armstrong